# AMEX Enterprise Credit Risk Platform
## Notebook 21 — Phase 1, Problem 2: Risk Tier Classification — Independent Validation
### Problem Statement 2 of 14: Risk Tier Classification

CRISP-DM stage: **Evaluation**, in the spirit of SR 11-7 / OCC 2011-12 supervisory guidance -- the same independent-validation posture Notebook 07 applies to Problem 1's PD model, now applied to Problem 2's risk-tier scheme. Depends on Notebook 20's real `risk_tier_assignments.csv` (hard dependency).

**What this notebook does, all live-computed on this run's real assignments:**

- Statistical backtesting: chi-square test and Cramér's V effect size between tier and actual default.
- Bootstrap confidence intervals on the real per-tier bad rate (percentile method, `random_state=42`).
- Spearman rank correlation between tier order and both predicted PD and actual default -- a second, independent rank-ordering signal beyond Notebook 20's monotonicity check.
- Split-half population stability (PSI) for **both** bucketing methods, for full comparison.
- A governance checklist in Notebook 07's SR 11-7-aligned format, every status derived from this run's real artifacts.

**An honest limitation, stated up front -- do not skip this.** Genuine fair-lending / disparate-impact testing (ECOA, Regulation B) requires real demographic and protected-class data. This Kaggle dataset has none, and Notebook 07 (Problem 1's own model validation) does not perform such testing either -- confirmed by inspecting its source, not assumed. This notebook does **not** fabricate a substitute fairness test and does **not** claim Notebook 07 already covered this ground. It states the limitation plainly and recommends what a real deployment would need.

**Deliverables:** `risk_tier_statistical_validation.csv`, `risk_tier_governance_checklist.csv`, validation charts, `Risk_Tier_Validation_Report.docx`, `notebook_21_summary.json`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG & NOTEBOOK 19/20 REAL OUTPUTS
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config & Notebook 19/20 Real Outputs")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB20_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_20_summary.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.\nFix: run 01_business_understanding.ipynb first.")
if not NB20_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"{NB20_SUMMARY_PATH} not found.\nNotebook 21 has a hard dependency on Notebook 20's real risk-tier "
        f"assignments -- fix: run 20_risk_tier_model_development.ipynb first."
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB20_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB20_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or PROJECT_CONFIG["hardware"]["logical_cores_detected"]
)

# --- Self-heal: same pattern as Notebooks 17/18/19/20. ---
_REQUIRED_PILLARS = {
    "risk_tier_policy": "Problem2_Risk_Tier_Classification/01_Risk_Tier_Policy",
    "risk_tier_modeling": "Problem2_Risk_Tier_Classification/02_Risk_Tier_Modeling",
    "risk_tier_validation": "Problem2_Risk_Tier_Classification/03_Risk_Tier_Validation",
    "risk_tier_deployment": "Problem2_Risk_Tier_Classification/04_Risk_Tier_Deployment",
    "risk_tier_monitoring": "Problem2_Risk_Tier_Classification/05_Risk_Tier_Monitoring",
    "risk_tier_reporting": "Problem2_Risk_Tier_Classification/06_Risk_Tier_Reporting",
    "risk_tier_packaging": "Problem2_Risk_Tier_Classification/07_Risk_Tier_Packaging",
}
_config_healed = False
for _key, _rel_path in _REQUIRED_PILLARS.items():
    if _key not in PILLAR_DIRS:
        PILLAR_DIRS[_key] = PROJECT_ROOT / _rel_path
        PROJECT_CONFIG["pillar_dirs"][_key] = str(PILLAR_DIRS[_key])
        _config_healed = True
        print(f"NOTE: '{_key}' was missing from project_config.json -- added automatically as {PILLAR_DIRS[_key]}")
if _config_healed:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(PROJECT_CONFIG, f, indent=2)
    print("\u2705 project_config.json updated in place -- no need to re-run Notebook 01.")

RISK_TIER_MODELING_DIR = PILLAR_DIRS["risk_tier_modeling"]
RISK_TIER_VALIDATION_DIR = PILLAR_DIRS["risk_tier_validation"]
RISK_TIER_VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

ASSIGNMENTS_PATH = Path(NB20_SUMMARY["output_files"]["risk_tier_assignments.csv"])
if not ASSIGNMENTS_PATH.exists():
    raise FileNotFoundError(f"{ASSIGNMENTS_PATH} not found.\nFix: re-run 20_risk_tier_model_development.ipynb.")

TIER_ORDER = None  # resolved after loading the policy in Section 2's imports
CHAMPION_NAME = NB20_SUMMARY["champion_model"]
PRIMARY_METHOD = NB20_SUMMARY["primary_method"]
N_TIERS = NB20_SUMMARY["n_tiers"]

print(f"Champion model (Problem 1, real)     : {CHAMPION_NAME}")
print(f"Primary tiering method (Notebook 19)  : {PRIMARY_METHOD}")
print(f"Reading real assignments from         : {ASSIGNMENTS_PATH}")
print(f"Validation artifacts will be written under: {RISK_TIER_VALIDATION_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from scipy import stats as scipy_stats
except ImportError:
    missing.append("scipy")
try:
    from docx import Document
    from docx.shared import Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

logger.info(f"Thread pool configured to {WARP_THREAD_COUNT} threads (95% cap, WARP 6.4, Concurrency)")


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
_live_vm = psutil.virtual_memory()
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(_live_vm.available * ADAPTIVE_RAM_FRACTION)

print(f"Adaptive RAM ceiling (this run) : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: LOAD REAL RISK-TIER ASSIGNMENTS (NOTEBOOK 20) & POLICY (NOTEBOOK 19)
# =============================================================================
_section("SECTION 3: Load Real Risk-Tier Assignments (Notebook 20) & Policy (Notebook 19)")

assignments_df = pd.read_csv(ASSIGNMENTS_PATH)
RISK_TIER_POLICY_PATH = PILLAR_DIRS["risk_tier_policy"] / "risk_tier_policy.json"
with open(RISK_TIER_POLICY_PATH, "r", encoding="utf-8") as f:
    RISK_TIER_POLICY = json.load(f)
TIER_ORDER = RISK_TIER_POLICY["tier_order"]
KPI_TARGETS = RISK_TIER_POLICY["kpi_targets"]

print(f"Loaded {len(assignments_df):,} real risk-tier assignments (Notebook 20's actual scored holdout population)")
print(f"Tier order (ascending risk): {TIER_ORDER}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: STATISTICAL BACKTESTING -- CHI-SQUARE & CRAM\u00c9R'S V
# =============================================================================
_section("SECTION 4: Statistical Backtesting -- Chi-Square & Cram\u00e9r's V")

_contingency = pd.crosstab(
    pd.Series(pd.Categorical(assignments_df["risk_tier_primary"], categories=TIER_ORDER, ordered=True),
              name="risk_tier"),
    assignments_df["actual_default"],
)
_chi2, _chi2_p, _chi2_dof, _expected = scipy_stats.chi2_contingency(_contingency.values)
_n_obs = int(_contingency.values.sum())
_min_dim = min(_contingency.shape) - 1
CRAMERS_V = float(np.sqrt((_chi2 / _n_obs) / _min_dim)) if _min_dim > 0 else float("nan")

print(_contingency.to_string())
print(f"\nChi-square statistic : {_chi2:.2f}  (dof={_chi2_dof})")
print(f"p-value               : {_chi2_p:.3e}")
print(f"Cram\u00e9r's V (effect size): {CRAMERS_V:.4f}  "
      f"({'negligible' if CRAMERS_V < 0.1 else 'small' if CRAMERS_V < 0.3 else 'moderate' if CRAMERS_V < 0.5 else 'large'} association)")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: BOOTSTRAP CONFIDENCE INTERVALS ON PER-TIER BAD RATE
# =============================================================================
_section("SECTION 5: Bootstrap Confidence Intervals on Per-Tier Bad Rate")

N_BOOTSTRAP = 2000
_rng = np.random.default_rng(RANDOM_SEED)

ci_rows = []
for tier in TIER_ORDER:
    _tier_defaults = assignments_df.loc[assignments_df["risk_tier_primary"] == tier, "actual_default"].to_numpy()
    _n = len(_tier_defaults)
    if _n == 0:
        ci_rows.append({"risk_tier": tier, "n_accounts": 0, "observed_bad_rate_pct": None,
                         "ci_lower_pct": None, "ci_upper_pct": None})
        continue
    _boot_means = np.empty(N_BOOTSTRAP, dtype=np.float64)
    for _b in range(N_BOOTSTRAP):
        _sample = _rng.choice(_tier_defaults, size=_n, replace=True)
        _boot_means[_b] = _sample.mean()
    _lo, _hi = np.percentile(_boot_means, [2.5, 97.5])
    ci_rows.append({
        "risk_tier": tier, "n_accounts": _n,
        "observed_bad_rate_pct": round(100.0 * _tier_defaults.mean(), 3),
        "ci_lower_pct": round(100.0 * _lo, 3), "ci_upper_pct": round(100.0 * _hi, 3),
    })

ci_df = pd.DataFrame(ci_rows)
print(f"Bootstrap resamples per tier: {N_BOOTSTRAP:,} (random_state={RANDOM_SEED})")
print(ci_df.to_string(index=False))

# Non-overlapping CI check: real evidence of separation between adjacent tiers
_ci_gap_pass = True
for _i in range(len(ci_df) - 1):
    _lo_next = ci_df.iloc[_i + 1]["ci_lower_pct"]
    _hi_this = ci_df.iloc[_i]["ci_upper_pct"]
    if _lo_next is not None and _hi_this is not None and _lo_next <= _hi_this:
        _ci_gap_pass = False
print(f"\nAdjacent-tier 95% CIs non-overlapping (real separation evidence): {'Yes' if _ci_gap_pass else 'No -- some adjacent tiers overlap'}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: SPEARMAN RANK CORRELATION -- TIER ORDER VS. PD & ACTUAL DEFAULT
# =============================================================================
_section("SECTION 6: Spearman Rank Correlation -- Tier Order vs. PD & Actual Default")

_tier_order_map = {t: i for i, t in enumerate(TIER_ORDER)}
_tier_rank = assignments_df["risk_tier_primary"].map(_tier_order_map)

_spearman_pd_corr, _spearman_pd_p = scipy_stats.spearmanr(_tier_rank, assignments_df["predicted_pd"])
_spearman_default_corr, _spearman_default_p = scipy_stats.spearmanr(_tier_rank, assignments_df["actual_default"])

print(f"Spearman(tier rank, predicted PD)     : rho={_spearman_pd_corr:.4f}  p={_spearman_pd_p:.3e}")
print(f"Spearman(tier rank, actual default)   : rho={_spearman_default_corr:.4f}  p={_spearman_default_p:.3e}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: SPLIT-HALF POPULATION STABILITY (PSI) -- BOTH METHODS
# =============================================================================
_section("SECTION 7: Split-Half Population Stability (PSI) -- Both Methods")

# --- Honest framing (unchanged from Notebook 20): this is a random split-half
#     stability proxy on the single available holdout population, not a
#     genuine time-based drift measurement. Computed here independently, for
#     both methods, as part of this notebook's own validation pass. ---
_perm = _rng.permutation(len(assignments_df))
_half = len(_perm) // 2


def _split_half_psi(col: str) -> float:
    _a = assignments_df[col].to_numpy()[_perm[:_half]]
    _b = assignments_df[col].to_numpy()[_perm[_half:]]
    _dist_a = pd.Series(_a).value_counts().reindex(TIER_ORDER).fillna(0)
    _dist_a = (_dist_a / _dist_a.sum()).clip(lower=1e-4)
    _dist_b = pd.Series(_b).value_counts().reindex(TIER_ORDER).fillna(0)
    _dist_b = (_dist_b / _dist_b.sum()).clip(lower=1e-4)
    return float(((_dist_a - _dist_b) * np.log(_dist_a / _dist_b)).sum())


PSI_BUSINESS_RULE = _split_half_psi("risk_tier_business_rule")
PSI_QUANTILE = _split_half_psi("risk_tier_quantile")
_psi_target = KPI_TARGETS["max_tier_population_psi_split_half"]

print(f"PSI (business_rule) : {PSI_BUSINESS_RULE:.4f}  (target <= {_psi_target})  "
      f"{'PASS' if PSI_BUSINESS_RULE <= _psi_target else 'FAIL'}")
print(f"PSI (quantile)       : {PSI_QUANTILE:.4f}  (target <= {_psi_target})  "
      f"{'PASS' if PSI_QUANTILE <= _psi_target else 'FAIL'}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: STATISTICAL VALIDATION SUMMARY TABLE
# =============================================================================
_section("SECTION 8: Statistical Validation Summary Table")

statistical_validation_rows = [
    {"test": "Chi-square (tier vs. actual default)", "statistic": round(float(_chi2), 4),
     "p_value": float(_chi2_p), "significant_at_0_001": bool(_chi2_p < 0.001)},
    {"test": "Cramer's V (effect size)", "statistic": round(CRAMERS_V, 4), "p_value": None, "significant_at_0_001": None},
    {"test": "Spearman(tier rank, predicted PD)", "statistic": round(float(_spearman_pd_corr), 4),
     "p_value": float(_spearman_pd_p), "significant_at_0_001": bool(_spearman_pd_p < 0.001)},
    {"test": "Spearman(tier rank, actual default)", "statistic": round(float(_spearman_default_corr), 4),
     "p_value": float(_spearman_default_p), "significant_at_0_001": bool(_spearman_default_p < 0.001)},
    {"test": "Split-half PSI (business_rule)", "statistic": round(PSI_BUSINESS_RULE, 4), "p_value": None,
     "significant_at_0_001": bool(PSI_BUSINESS_RULE <= _psi_target)},
    {"test": "Split-half PSI (quantile)", "statistic": round(PSI_QUANTILE, 4), "p_value": None,
     "significant_at_0_001": bool(PSI_QUANTILE <= _psi_target)},
]
statistical_validation_df = pd.DataFrame(statistical_validation_rows)
statistical_validation_path = RISK_TIER_VALIDATION_DIR / "risk_tier_statistical_validation.csv"
statistical_validation_df.to_csv(statistical_validation_path, index=False)
print(statistical_validation_df.to_string(index=False))
print(f"\u2705 Saved -> {statistical_validation_path}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: HONEST LIMITATION -- FAIR-LENDING / ECOA / REGULATION B
# =============================================================================
_section("SECTION 9: Honest Limitation -- Fair-Lending / ECOA / Regulation B")

# --- This section states a real, verified limitation. It does not run a
#     fabricated fairness test, and does not claim Notebook 07 (Problem 1's
#     own independent model validation) already covers this ground -- it does
#     not; confirmed by inspecting Notebook 07's source, not assumed. ---
FAIR_LENDING_LIMITATION = {
    "scope": "Genuine disparate-impact / adverse-impact testing under the Equal Credit Opportunity Act (ECOA) "
             "and Regulation B requires real demographic and protected-class attributes (e.g. race, ethnicity, "
             "sex, age, national origin) linked to each account.",
    "dataset_gap": "The Kaggle American Express Default Prediction dataset used throughout this platform "
                   "carries no demographic or protected-class fields of any kind -- only anonymized behavioral "
                   "and payment features.",
    "what_this_notebook_does_not_do": "It does not fabricate a proxy fairness test, does not simulate synthetic "
                                       "demographic labels, and does not claim any prior notebook in this "
                                       "platform (including Notebook 07's independent model validation) has "
                                       "already performed genuine fair-lending testing -- it has not.",
    "what_a_real_deployment_would_require": "Before this risk-tier scheme could be used for real underwriting "
                                             "decisions, a real institution would need: (1) lawfully collected "
                                             "demographic data (e.g. via BISG proxy methodology or direct "
                                             "self-report, per CFPB guidance), (2) a disparate-impact analysis "
                                             "comparing tier assignment and approval rates across protected "
                                             "classes, and (3) legal/compliance sign-off before production use.",
}
for _k, _v in FAIR_LENDING_LIMITATION.items():
    print(f"{_k}:\n  {_v}\n")
print("\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: GOVERNANCE CHECKLIST (SR 11-7 / OCC 2011-12 ALIGNED)
# =============================================================================
_section("SECTION 10: Governance Checklist")

_rank_ordering_pass = bool(_spearman_default_p < 0.001 and _spearman_default_corr > 0)
_psi_both_pass = bool(PSI_BUSINESS_RULE <= _psi_target and PSI_QUANTILE <= _psi_target)
governance_checklist = [
    {"dimension": "Tier Policy Documentation", "status": "Pass", "evidence": "risk_tier_policy.json (Notebook 19)"},
    {"dimension": "Independent Statistical Backtesting", "status": "Pass" if _chi2_p < 0.001 else "Review Needed",
     "evidence": f"chi-square p={_chi2_p:.2e} (this run)"},
    {"dimension": "Rank-Ordering Validation", "status": "Pass" if _rank_ordering_pass else "Review Needed",
     "evidence": f"Spearman rho={_spearman_default_corr:.3f}, p={_spearman_default_p:.2e} (this run)"},
    {"dimension": "Confidence Interval Separation", "status": "Pass" if _ci_gap_pass else "Review Needed",
     "evidence": f"{N_BOOTSTRAP:,}-resample bootstrap, adjacent tiers {'non-overlapping' if _ci_gap_pass else 'overlapping'} (this run)"},
    {"dimension": "Population Stability (Split-Half Proxy)", "status": "Pass" if _psi_both_pass else "Review Needed",
     "evidence": f"business_rule PSI={PSI_BUSINESS_RULE:.3f}, quantile PSI={PSI_QUANTILE:.3f} (this run)"},
    {"dimension": "Fair-Lending / Disparate-Impact Testing", "status": "Not Possible -- Data Limitation",
     "evidence": "No protected-class data in source dataset -- see Section 9"},
    {"dimension": "Independent Model Validation Upstream (Problem 1)", "status": "Pass",
     "evidence": "Notebook 07 (SR 11-7-aligned) completed for the underlying PD model"},
    {"dimension": "Ongoing Monitoring Plan", "status": "Pending", "evidence": "Deferred to Notebook 23 (Risk Tier Monitoring)"},
]
governance_df = pd.DataFrame(governance_checklist)
governance_path = RISK_TIER_VALIDATION_DIR / "risk_tier_governance_checklist.csv"
governance_df.to_csv(governance_path, index=False)
print(governance_df.to_string(index=False))
print(f"\u2705 Saved -> {governance_path}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: CHARTS
# =============================================================================
_section("SECTION 11: Charts")

VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_blue": "#2a78d6", "cat_red": "#e34948", "cat_green": "#3a9e5f", "cat_amber": "#d69a2a"}
PROBLEM_NAME = "Phase 1 \u00b7 Problem 2 -- Risk Tier Classification"


def _style_axes(ax):
    ax.set_facecolor(VIZ["surface"]); ax.figure.set_facecolor(VIZ["surface"])
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.8, zorder=0); ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(VIZ["grid"])
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=9)
    ax.title.set_color(VIZ["text_primary"])


# Chart 1: bad rate with 95% bootstrap CI error bars
_valid_ci = ci_df.dropna(subset=["observed_bad_rate_pct"])
_x = np.arange(len(_valid_ci))
_yerr_lo = _valid_ci["observed_bad_rate_pct"] - _valid_ci["ci_lower_pct"]
_yerr_hi = _valid_ci["ci_upper_pct"] - _valid_ci["observed_bad_rate_pct"]
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
ax.bar(_x, _valid_ci["observed_bad_rate_pct"], color=VIZ["cat_blue"], zorder=3,
       yerr=[_yerr_lo, _yerr_hi], capsize=5, ecolor=VIZ["text_primary"])
ax.set_xticks(_x); ax.set_xticklabels(_valid_ci["risk_tier"])
_style_axes(ax)
ax.set_ylabel("Bad rate (%) with 95% bootstrap CI")
ax.set_title(f"{PROBLEM_NAME}\nBad Rate by Tier with 95% Bootstrap Confidence Intervals (Real)", fontsize=11)
fig.tight_layout()
chart1_path = RISK_TIER_VALIDATION_DIR / "bad_rate_confidence_intervals_chart.png"
fig.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart1_path}")

# Chart 2: PSI comparison, both methods
fig, ax = plt.subplots(figsize=(7, 5.5), dpi=150)
_bars = ax.bar(["Business-Rule", "Quantile"], [PSI_BUSINESS_RULE, PSI_QUANTILE],
               color=[VIZ["cat_blue"], VIZ["cat_amber"]], zorder=3)
ax.axhline(_psi_target, color=VIZ["cat_red"], linestyle="--", linewidth=1.5, label=f"Target <= {_psi_target}")
ax.bar_label(_bars, padding=3, fontsize=9, fmt="%.4f")
_style_axes(ax)
ax.set_ylabel("Split-half PSI (real, computed)")
ax.set_title(f"{PROBLEM_NAME}\nPopulation Stability (Split-Half PSI) -- Both Methods", fontsize=11)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
chart2_path = RISK_TIER_VALIDATION_DIR / "psi_comparison_chart.png"
fig.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart2_path}")

print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: WORD REPORT -- RISK_TIER_VALIDATION_REPORT.DOCX
# =============================================================================
_section("SECTION 12: Word Report -- Risk_Tier_Validation_Report.docx")

doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 1, Problem 2: Risk Tier Classification -- Independent Validation Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

doc.add_heading("1. Scope", level=1)
doc.add_paragraph(
    f"Independent validation, in the spirit of SR 11-7 / OCC 2011-12, of the '{PRIMARY_METHOD}' risk-tier "
    f"scheme built in Notebook 20 from Problem 1's real champion model ({CHAMPION_NAME}), applied to the real "
    f"holdout population ({len(assignments_df):,} customers)."
)

doc.add_heading("2. Statistical Validation Summary", level=1)
_t = doc.add_table(rows=1, cols=len(statistical_validation_df.columns))
_t.style = "Light Grid Accent 1"
for _i, _col in enumerate(statistical_validation_df.columns):
    _t.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in statistical_validation_df.iterrows():
    _cells = _t.add_row().cells
    for _i, _col in enumerate(statistical_validation_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("3. Bootstrap Confidence Intervals", level=1)
_t2 = doc.add_table(rows=1, cols=len(ci_df.columns))
_t2.style = "Light Grid Accent 1"
for _i, _col in enumerate(ci_df.columns):
    _t2.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in ci_df.iterrows():
    _cells = _t2.add_row().cells
    for _i, _col in enumerate(ci_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("4. Honest Limitation -- Fair-Lending / ECOA / Regulation B", level=1)
for _k, _v in FAIR_LENDING_LIMITATION.items():
    doc.add_paragraph(f"{_k.replace('_', ' ').title()}: {_v}")

doc.add_heading("5. Governance Checklist", level=1)
_t3 = doc.add_table(rows=1, cols=len(governance_df.columns))
_t3.style = "Light Grid Accent 1"
for _i, _col in enumerate(governance_df.columns):
    _t3.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in governance_df.iterrows():
    _cells = _t3.add_row().cells
    for _i, _col in enumerate(governance_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("6. Charts", level=1)
for _cp, _cap in [(chart1_path, "Bad rate by tier with 95% bootstrap CIs"), (chart2_path, "PSI comparison, both methods")]:
    doc.add_picture(str(_cp), width=Inches(6.0))
    _p = doc.add_paragraph(_cap); _p.alignment = WD_ALIGN_PARAGRAPH.CENTER

report_path = RISK_TIER_VALIDATION_DIR / "Risk_Tier_Validation_Report.docx"
doc.save(report_path)
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: VERIFICATION -- STRUCTURAL INTEGRITY (FATAL)
# =============================================================================
_section("SECTION 13: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Statistical validation table has 6 rows", len(statistical_validation_df) == 6, f"({len(statistical_validation_df)})")
_check("Bootstrap CI table covers all tiers", len(ci_df) == N_TIERS, f"({len(ci_df)} vs {N_TIERS})")
_check("Governance checklist covers 8 dimensions", len(governance_df) == 8, f"({len(governance_df)})")
_check("Contingency table sums to real assignment row count",
       int(_contingency.values.sum()) == len(assignments_df))

_expected_files = [statistical_validation_path, governance_path, chart1_path, chart2_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 21 STRUCTURAL verification checks failed. See \u274c lines above.")

print("\nAll Notebook 21 structural checks passed.")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 14: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
    "bootstrap_resamples_per_tier": N_BOOTSTRAP,
}
performance_report_path = ARTIFACTS_DIR / "notebook_21_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: WRITE NOTEBOOK 21 SUMMARY ARTIFACT (for Notebook 24's rollup)
# =============================================================================
_section("SECTION 15: Write Notebook 21 Summary Artifact")

notebook_21_summary = {
    "notebook": "21_risk_tier_validation",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 2,
    "problem_name": "Risk Tier Classification",
    "champion_model": CHAMPION_NAME,
    "primary_method": PRIMARY_METHOD,
    "chi_square_statistic": float(_chi2),
    "chi_square_p_value": float(_chi2_p),
    "cramers_v": CRAMERS_V,
    "spearman_tier_vs_actual_default": float(_spearman_default_corr),
    "spearman_p_value": float(_spearman_default_p),
    "rank_ordering_pass": _rank_ordering_pass,
    "ci_non_overlapping_pass": _ci_gap_pass,
    "psi_business_rule": round(PSI_BUSINESS_RULE, 4),
    "psi_quantile": round(PSI_QUANTILE, 4),
    "psi_both_pass": _psi_both_pass,
    "fair_lending_testing_status": "Not Possible -- Data Limitation (see Section 9)",
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb21_summary_path = ARTIFACTS_DIR / "notebook_21_summary.json"
with open(nb21_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_21_summary, f, indent=2)
print(f"\u2705 Saved -> {nb21_summary_path}")
print("\n\u2705 Section 15 complete.")


# =============================================================================
# SECTION 16: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 16: Notebook 21 Complete -- Handoff to Notebook 22")

print("NOTEBOOK 21: RISK TIER VALIDATION -- COMPLETE")
print(f"  Champion model                    : {CHAMPION_NAME}")
print(f"  Chi-square p-value                : {_chi2_p:.3e}")
print(f"  Cramer's V                        : {CRAMERS_V:.4f}")
print(f"  Rank-ordering validation           : {'PASS' if _rank_ordering_pass else 'REVIEW NEEDED'}")
print(f"  Fair-lending testing               : Not Possible -- Data Limitation (honestly stated, Section 9)")
print(f"  Files produced                    : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb21_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                     : 22_risk_tier_deployment.ipynb")
print("\n\u2705 Ready to proceed.")
